**Elastic Load Balancing (ELB)** and **Auto Scaling** are the two services that work together to deliver high availability and elasticity for EC2-based architectures. ELB distributes incoming traffic across multiple targets; Auto Scaling adjusts the number of targets based on demand. Together they are the foundation of every scalable, fault-tolerant AWS application.

## Why a Load Balancer?

A load balancer sits in front of your instances and:
- Distributes traffic across multiple instances to prevent any single one from being overwhelmed
- Performs **health checks** and stops sending traffic to unhealthy instances
- Provides a **single DNS endpoint** — clients never need to know about individual instance IPs
- Enables **zero-downtime deployments** by draining connections before removing instances
- Terminates **SSL/TLS** so instances don't need to handle certificate management
- Enforces **stickiness** so a client always hits the same instance (where needed)

## ELB Types

AWS offers four load balancer types. Three are current; one is legacy.

| | ALB | NLB | GWLB | CLB |
|---|---|---|---|---|
| **Layer** | 7 (HTTP/HTTPS/gRPC) | 4 (TCP/UDP/TLS) | 3 (IP) | 4/7 (legacy) |
| **Routing** | Content-based (path, host, headers) | Connection-based | Transparent pass-through | Round-robin |
| **Static IP** | No (DNS only) | Yes (per AZ) | Yes | No |
| **Performance** | High | Ultra-low latency | — | Lower |
| **Use case** | Web apps, microservices, APIs | High-performance TCP/UDP, firewalls | Security appliances | Legacy only |

> **Exam tip:** Never choose CLB for a new design. ALB or NLB for everything new.

## Application Load Balancer (ALB)

ALB operates at **Layer 7** — it understands HTTP and HTTPS. This lets it make routing decisions based on the content of the request.

### Components

```
Client
  │
  ▼
Listener (port 443, HTTPS)
  │
  ├── Rule 1: path /api/*  ──────► Target Group: api-servers
  ├── Rule 2: host api.example.com ► Target Group: api-servers
  ├── Rule 3: header X-Version: v2 ► Target Group: v2-servers
  └── Default Rule ────────────── ► Target Group: web-servers
```

| Component | Description |
|---|---|
| **Listener** | Checks for connection requests on a port (80, 443) |
| **Rule** | Conditions (path, host, headers, query string, source IP) + action |
| **Target Group** | A group of targets (EC2 instances, IPs, Lambda, other ALBs) |
| **Target** | The individual EC2 instance, IP, or Lambda function receiving traffic |

### ALB routing rules — actions
- **Forward** to a target group
- **Weighted forward** — split traffic between target groups (A/B testing, canary deployments)
- **Redirect** — e.g. HTTP → HTTPS, or one URL to another
- **Fixed response** — return a static HTTP response (useful for maintenance pages)

### ALB features
- **SSL termination** — certificate managed by ACM (AWS Certificate Manager)
- **Sticky sessions** — via a load balancer-generated cookie (`AWSALB`); duration configurable
- **gRPC support** — natively routes gRPC traffic
- **WebSocket support** — maintains long-lived connections
- **X-Forwarded-For header** — passes the original client IP to backend instances
- **WAF integration** — attach AWS WAF for Layer 7 protection

## Network Load Balancer (NLB)

NLB operates at **Layer 4** (TCP/UDP). It does not inspect the content of traffic — it simply forwards packets based on IP and port. This makes it extremely fast with latencies in the single-digit milliseconds.

### Key characteristics

| Feature | Detail |
|---|---|
| **Static IPs** | One static IP per AZ (or Elastic IP); useful for IP whitelisting |
| **Preserves source IP** | Client IP is passed through to the target (ALB rewrites it) |
| **Ultra-high throughput** | Handles millions of requests per second |
| **TLS passthrough** | Can pass encrypted traffic directly to the target without terminating it |
| **UDP support** | Only load balancer type that supports UDP (DNS, gaming, IoT) |

### When to choose NLB over ALB
- You need a **static IP** or Elastic IP on the load balancer
- Traffic is **non-HTTP** — TCP, UDP, custom protocol
- You need the absolute **lowest latency** (financial systems, real-time gaming)
- You need to preserve the **client's source IP** at the instance level

## Gateway Load Balancer (GWLB)

GWLB operates at **Layer 3** (IP). It is designed specifically to deploy, scale, and manage third-party network appliances — firewalls, intrusion detection systems, deep packet inspection tools.

```
Internet → GWLB → Firewall Appliance Fleet → GWLB → Application
```

Traffic is transparently bumped to the appliance fleet and returned, without the source or destination knowing. GWLB uses the **GENEVE** protocol on port 6081 to encapsulate and route traffic.

> GWLB is primarily an exam topic — you are unlikely to configure it in a typical application architecture, but you need to recognise it when a question involves inline network appliances.

## Cross-AZ Load Balancing, Health Checks & Draining

### Cross-zone load balancing
By default, each load balancer node distributes traffic only to targets in its own AZ. With cross-zone load balancing enabled, each node distributes traffic evenly across all targets in all AZs.

| | ALB | NLB | GWLB |
|---|---|---|---|
| Cross-zone default | **On** (free) | **Off** (charged if enabled) | **Off** (charged if enabled) |

### Health checks
Each target group has a health check configuration:
- **Protocol + path** — e.g. HTTP `/health`
- **Healthy threshold** — consecutive successes to mark healthy (default: 5)
- **Unhealthy threshold** — consecutive failures to mark unhealthy (default: 2)
- **Interval** — how often to check (default: 30 seconds)

An unhealthy target stops receiving new requests until it passes health checks again.

### Connection draining (Deregistration delay)
When an instance is deregistered from a target group (or marked unhealthy), ELB stops sending new requests to it but waits for in-flight requests to complete. This period is the **deregistration delay** — default 300 seconds, configurable from 0 to 3600. Set it low (e.g. 30 seconds) for stateless apps; keep it higher for long-lived requests.

## Auto Scaling Groups (ASG)

An **Auto Scaling Group** manages a fleet of EC2 instances, automatically adjusting the count based on demand or a schedule.

### Core settings

| Setting | Description |
|---|---|
| **Minimum capacity** | Floor — ASG will never go below this count |
| **Desired capacity** | Target count — ASG tries to maintain this |
| **Maximum capacity** | Ceiling — ASG will never exceed this count |
| **Launch template** | Defines the instance config: AMI, type, key pair, SG, user data |
| **VPC + subnets** | ASG distributes instances across the selected subnets (AZs) |

### Launch template vs launch configuration
Always use **launch templates** — they support versioning, mixed instance types, Spot + On-Demand combinations, and all current EC2 features. Launch configurations are legacy and cannot be edited; they are being deprecated.

### ASG and load balancers
Attach an ASG to one or more target groups. New instances are automatically registered when launched; terminated instances are automatically deregistered. The ASG respects the target group's health checks — if ELB marks an instance unhealthy, the ASG can replace it.

## Scaling Policies

### 1. Target Tracking Scaling
The simplest and most recommended policy. You pick a metric and a target value — AWS automatically adds or removes instances to keep the metric at that target.

Examples:
- Keep average CPU at 50%
- Keep `RequestCountPerTarget` at 1000 per instance
- Keep average network in at 1 GB/s

AWS manages both scale-out and scale-in automatically.

### 2. Step Scaling
Define CloudWatch alarms and specify how many instances to add or remove based on the magnitude of the alarm breach.

```
CPU 50–70% → add 1 instance
CPU 70–90% → add 2 instances
CPU > 90%  → add 4 instances
```

More granular than target tracking. Useful when you need different scaling aggressiveness at different thresholds.

### 3. Simple Scaling (legacy)
Like step scaling but with only one step and a cooldown period during which no further scaling happens. Largely superseded by step and target tracking.

### 4. Scheduled Scaling
Scale based on a known schedule — not a metric. Set desired/min/max at specific times.

```
Every weekday 08:00 UTC → min=10, desired=20
Every weekday 20:00 UTC → min=2,  desired=5
```

Use for predictable traffic patterns — business hours, end-of-month batch runs.

### 5. Predictive Scaling
Uses ML to analyse historical CloudWatch data and forecast future load. Pre-scales the ASG *before* the demand arrives, rather than reacting after the fact. Combine with target tracking for best results.

### Cooldown period
After a scaling activity, the ASG waits for the cooldown (default 300 seconds) before evaluating another scale-out trigger. Prevents rapid, repeated scaling. Not applicable to target tracking (it has its own internal stabilisation).

## ASG Health Checks & Instance Refresh

### Health check types

| Type | What it checks |
|---|---|
| **EC2** (default) | Instance is running and passes EC2 status checks |
| **ELB** | Instance passes the target group's application health check |
| **Custom** | Your own health check via the `SetInstanceHealth` API |

Enable ELB health checks on your ASG — otherwise an instance that is running but returning 500 errors is still considered healthy by the ASG.

### Instance Refresh
When you update the launch template (e.g. new AMI), use **Instance Refresh** to roll out the update across the fleet without manual work.

- Specify a **minimum healthy percentage** (e.g. 90%) — ASG replaces instances in batches while keeping at least 90% healthy
- Optionally specify a **warm-up time** before a new instance is counted as healthy
- Can be paused or cancelled mid-way

### Lifecycle Hooks
Lifecycle hooks pause the instance at a transition point (launch or terminate) to allow custom actions.

```
Scale-out:  Pending → [Pending:Wait] → Pending:Proceed → InService
                             ↑
                     Run bootstrap, register with service discovery,
                     run tests — then call complete-lifecycle-action

Scale-in:   InService → [Terminating:Wait] → Terminating:Proceed → Terminated
                                ↑
                     Drain connections, upload logs, deregister —
                     then call complete-lifecycle-action
```

The instance waits up to the configured timeout (default 1 hour, max 48 hours) before proceeding.

## Warm Pools

A **Warm Pool** maintains a pool of pre-initialised, stopped (or running) instances ready to serve traffic immediately when the ASG needs to scale out. This eliminates the bootstrapping delay during scale-out events.

```
Without warm pool:  scale-out trigger → launch → bootstrap (2–10 min) → InService
With warm pool:     scale-out trigger → move from pool → InService (seconds)
```

Warm pool instances can be kept in `Stopped` state (cost-effective — pay for EBS only) or `Running` state (faster, pay for compute).

## Working with ELB and Auto Scaling via boto3

In [ ]:
import boto3

elbv2 = boto3.client('elbv2', region_name='us-east-1')
asg_client = boto3.client('autoscaling', region_name='us-east-1')

In [ ]:
# Create an Application Load Balancer
alb = elbv2.create_load_balancer(
    Name='demo-alb',
    Subnets=['subnet-aaa', 'subnet-bbb'],   # one per AZ
    SecurityGroups=['sg-xxxxxxxxx'],
    Scheme='internet-facing',
    Type='application',
    IpAddressType='ipv4'
)
alb_arn = alb['LoadBalancers'][0]['LoadBalancerArn']
print(f"ALB ARN: {alb_arn}")
print(f"DNS:     {alb['LoadBalancers'][0]['DNSName']}")

In [ ]:
# Create a target group with HTTP health checks
tg = elbv2.create_target_group(
    Name='demo-web-tg',
    Protocol='HTTP',
    Port=80,
    VpcId='vpc-xxxxxxxxx',
    HealthCheckPath='/health',
    HealthCheckIntervalSeconds=30,
    HealthyThresholdCount=2,
    UnhealthyThresholdCount=3,
    TargetType='instance'
)
tg_arn = tg['TargetGroups'][0]['TargetGroupArn']
print(f"Target Group ARN: {tg_arn}")

In [ ]:
# Create an HTTPS listener that forwards to the target group
elbv2.create_listener(
    LoadBalancerArn=alb_arn,
    Protocol='HTTPS',
    Port=443,
    Certificates=[{'CertificateArn': 'arn:aws:acm:us-east-1:123456789012:certificate/xxx'}],
    DefaultActions=[{'Type': 'forward', 'TargetGroupArn': tg_arn}]
)

# Redirect HTTP → HTTPS
elbv2.create_listener(
    LoadBalancerArn=alb_arn,
    Protocol='HTTP',
    Port=80,
    DefaultActions=[{
        'Type': 'redirect',
        'RedirectConfig': {'Protocol': 'HTTPS', 'Port': '443', 'StatusCode': 'HTTP_301'}
    }]
)
print("Listeners created")

In [ ]:
# Create an Auto Scaling Group attached to the target group
asg_client.create_auto_scaling_group(
    AutoScalingGroupName='demo-asg',
    LaunchTemplate={
        'LaunchTemplateName': 'demo-lt',
        'Version': '$Latest'
    },
    MinSize=2,
    DesiredCapacity=2,
    MaxSize=10,
    TargetGroupARNs=[tg_arn],
    HealthCheckType='ELB',
    HealthCheckGracePeriod=120,
    VPCZoneIdentifier='subnet-aaa,subnet-bbb'
)
print("ASG created")

In [ ]:
# Attach a target tracking policy: keep average CPU at 50%
asg_client.put_scaling_policy(
    AutoScalingGroupName='demo-asg',
    PolicyName='cpu-target-tracking',
    PolicyType='TargetTrackingScaling',
    TargetTrackingConfiguration={
        'PredefinedMetricSpecification': {
            'PredefinedMetricType': 'ASGAverageCPUUtilization'
        },
        'TargetValue': 50.0,
        'DisableScaleIn': False
    }
)
print("Target tracking policy attached")

In [ ]:
# Describe the ASG — see current instance states
response = asg_client.describe_auto_scaling_groups(
    AutoScalingGroupNames=['demo-asg']
)
asg = response['AutoScalingGroups'][0]
print(f"Min: {asg['MinSize']}  Desired: {asg['DesiredCapacity']}  Max: {asg['MaxSize']}")
print(f"Instances:")
for inst in asg['Instances']:
    print(f"  {inst['InstanceId']}  {inst['LifecycleState']}  {inst['HealthStatus']}")

## Summary

| Concept | Key Takeaway |
|---|---|
| ALB | Layer 7; content-based routing by path, host, headers; ideal for HTTP/HTTPS microservices |
| NLB | Layer 4; static IP per AZ; ultra-low latency; use for TCP/UDP or IP whitelisting |
| GWLB | Layer 3; transparent bump for inline network appliances (firewalls, IDS) |
| Target group | Pool of targets; has its own health check; supports EC2, IP, Lambda |
| Deregistration delay | Grace period for in-flight requests when an instance is removed |
| Cross-zone LB | ALB: on by default; NLB/GWLB: off by default (extra cost to enable) |
| ASG | Manages fleet size; distributes across AZs; integrates with ELB target groups |
| Launch template | Always prefer over launch configuration — supports versioning and mixed instances |
| Target tracking | Simplest scaling policy — set a metric target, AWS handles scale-out and scale-in |
| Step scaling | Fine-grained control — different actions at different alarm thresholds |
| Scheduled scaling | Predictable patterns — scale by time, not metrics |
| Predictive scaling | ML-driven pre-scaling based on historical patterns |
| ELB health check type | Always enable on ASG — catches app-level failures, not just EC2 status |
| Instance Refresh | Rolling AMI update across the fleet without manual intervention |
| Lifecycle hooks | Pause instances at launch/terminate to run custom actions |
| Warm pools | Pre-initialised instances for near-instant scale-out |